In [29]:
#utils
import pandas as pd
import numpy as np
import contextlib
import os
import warnings
from scipy.stats import norm
import sys

#visualization libraries
import matplotlib.pyplot as plt

#machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve

#models 
from models.AutoEncoder import Autoencoder
from models.ContractiveAutoEncoder import ContractiveAutoEncoder
from models.VariationalAutoencoder import VariationalAutoEncoder
from models.alad.ALAD import ALAD


warnings.filterwarnings("ignore")

In [30]:
df = pd.read_csv('SimulationData.csv')
df = df.head(1000)

In [31]:
target_feature = "isghost"
columns = df.drop(columns=target_feature).columns
#instantiate two more datasets one to contain the normal data
#and the other to hold the anomalous instances
train_normal_data = df[df[target_feature] == 0].drop(columns=target_feature)
anomalous_data = df[df[target_feature] == 1].drop(columns=target_feature)

In [32]:
def get_normal_data(df):
    df = df.sample(frac=1)
    features = df.drop([target_feature], axis=1)
    label = df[target_feature]

    #the stratify parameter here is set to be "isghost" meaning that the 
    #proportions of ghost and normal tracks in both train and test datasets
    #will be kept the same...
    x_train, x_test, y_train, y_test = train_test_split(
        features, label, test_size=0.2, stratify=label
    )

    #remember that at least for the autoencoders we're gonna use just the 
    #normal instances during the training phase so we have to filter those in x_train
    train_index = y_train[y_train == 0].index
    train_normal_data = x_train.loc[train_index]

    return train_normal_data, x_train, y_train, x_test, y_test

In [33]:
def standardize_normalize(train_x, normal_train_data, test_data):


    non_zero_cols = train_x.columns[(train_x != 0).any()]

    # Calculate mean and standard deviation only for non-zero columns
    mean = train_x[non_zero_cols].mean()
    std = train_x[non_zero_cols].std() + sys.float_info.min

    # Standardize the non-zero columns in normal_train_data, train_x, and test_data
    normal_train_data[non_zero_cols] = (normal_train_data[non_zero_cols] - mean) / std
    train_x[non_zero_cols] = (train_x[non_zero_cols] - mean) / std
    test_data[non_zero_cols] = (test_data[non_zero_cols] - mean) / std

    # Create a scaler and apply Min-Max scaling to all columns
    scaler = MinMaxScaler()
    normal_train_data = scaler.fit_transform(normal_train_data)
    train_x = scaler.transform(train_x)
    test_data = scaler.transform(test_data)
    
    return train_x, normal_train_data, test_data

In [34]:
train_normal_data, x_train, y_train, x_test, y_test = get_normal_data(df)
x_train, train_normal_data, x_test = standardize_normalize(x_train, train_normal_data, x_test)

In [35]:
INPUT_DIM = train_normal_data.shape[1]
ENCODING_DIM = 3
learing_rates = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05]

In [36]:
def get_data():

    train_normal_data, x_train, _, x_test, y_test = get_normal_data(df.sample(frac=1).reset_index(drop=True))
    _, train_normal_data, x_test = standardize_normalize(x_train, train_normal_data, x_test)

    return train_normal_data, x_test, y_test

In [37]:
def build_models(learning_rate, encoding_dim):

    ae = Autoencoder(INPUT_DIM, encoding_dim, learning_rate)
    cae = ContractiveAutoEncoder(INPUT_DIM, encoding_dim, lambda_=0.01, learning_rate = learning_rate)
    vae = VariationalAutoEncoder(INPUT_DIM, encoding_dim, learning_rate)
    alad = ALAD(input_dim=INPUT_DIM, latent_dim=encoding_dim, learning_rate=learning_rate)

    ae.build(input_shape=(train_normal_data.shape))
    cae.build(input_shape=(train_normal_data.shape))
    vae.build(input_shape=(train_normal_data.shape))
    
    return ae, cae, vae, alad

In [38]:

def avgNestedLists(nested_vals):
    """
    Averages a 2-D array and returns a 1-D array of all of the columns
    averaged together, regardless of their dimensions.
    """
    output = []
    maximum = 0
    for lst in nested_vals:
        if len(lst) > maximum:
            maximum = len(lst)
    for index in range(maximum): # Go through each index of longest list
        temp = []
        for lst in nested_vals: # Go through each list
            if index < len(lst): # If not an index error
                temp.append(lst[index])
        output.append(np.nanmean(temp))
    return output

In [39]:
import numpy as np
from sklearn.metrics import roc_curve, auc
from scipy import stats


def mean_and_confidence(model, threshold_function, alpha=0.95, iterations=2, transfer=False, epochs=2, learning_rate = 0.003):
   

    #initialize all the containers for teh metrics to track
    list_fpr = []
    list_tpr = []
    list_auc = []

    #precision and recall will be used for 
    #the curve but also as general parameters 
    list_precision_curve = []
    list_recall_curve = []

    #classification metrics 
    list_accuracy = []
    list_precision = []
    list_recall = []
    list_f1 = []

    list_training_time = []

    model.save_weights('saved_weights/exp2/my_model' +  model.model_name)

    

    # Perform 100 iterations
    for _ in range(iterations):

        x_train, x_test, y_test = get_data()

        history, training_time = model.train(x_train, epochs=epochs, batch_size=30, validation_split=0.2)
        ###time each model 
        threshold = threshold_function(x_train, x_test, y_test)
        predictions = model.find_anomalies(x_test, threshold)
        accuracy, precision, recall, f1 = model.classification_metrics(y_test, predictions)

        #re-set all the trainable variables of the model
        model.load_weights('saved_weights/exp2/my_model' +  model.model_name)
        
        fpr_ae, tpr_ae, _ = roc_curve(y_test, predictions)
        auc_ae = auc(fpr_ae, tpr_ae)
        list_fpr.append(fpr_ae)
        list_tpr.append(tpr_ae)
        

        precision_curve, recall_curve, _ = precision_recall_curve(y_test, predictions)
        list_precision_curve.append(precision_curve)
        list_recall_curve.append(recall_curve)


        list_accuracy.append(accuracy)
        list_precision.append(precision)
        list_recall.append(recall)
        list_f1.append(f1)
        list_training_time.append(training_time)
        list_auc.append(auc_ae)

    #roc_curve can return different sized arrays
    #this means that ther needs to be a custom function
    #to compute the mean 
    avg_fpr = avgNestedLists(list_fpr)
    avg_tpr = avgNestedLists(list_tpr)
    avg_precision_curve = avgNestedLists(list_precision_curve)
    avg_recall_curve = avgNestedLists(list_recall_curve)
    avg_acc = np.mean(list_accuracy)
    avg_rec = np.mean(list_recall)
    avg_pre = np.mean(list_precision)
    avg_f1 = np.mean(list_f1)
    avg_time = np.mean(list_training_time)
    avg_auc = np.mean(list_auc)


    #standard deviations
    accuracy_std = np.std(list_accuracy)
    precision_std = np.std(list_precision)
    recall_std = np.std(list_recall)
    f1_std = np.std(list_f1)
    training_time_std = np.std(list_training_time)
    auc_std = np.std(list_auc)

    #how to take the "mean curve"  
    interval_acc = stats.t.interval(alpha=alpha, loc=np.mean(list_accuracy), df=len(list_accuracy)-1, scale=stats.sem(list_accuracy))
    interval_rec = stats.t.interval(alpha=alpha, loc=np.mean(list_recall), df=len(list_recall)-1, scale=stats.sem(list_recall))
    interval_pre = stats.t.interval(alpha=alpha, loc=np.mean(list_precision), df=len(list_precision)-1, scale=stats.sem(list_precision))
    interval_f1 = stats.t.interval(alpha=alpha, loc=np.mean(list_f1), df=len(list_f1)-1, scale=stats.sem(list_f1))
    interval_time = stats.t.interval(alpha=alpha, loc=np.mean(list_training_time), df=len(list_training_time)-1, scale=stats.sem(list_training_time))
    interval_auc = stats.t.interval(alpha=alpha, loc=np.mean(list_auc), df=len(list_auc)-1, scale=stats.sem(list_auc))




    averages = {
        "Average FPR": avg_fpr,
        "Average TPR": avg_tpr,
        "Average precision curve": avg_precision_curve,
        "Average recall curve": avg_recall_curve,
        "Average accuracy": avg_acc,
        "Average recall": avg_rec,
        "Average precision": avg_pre,
        "Average F-1": avg_f1,
        "Average training time": avg_time,
        "Average AUC": avg_auc,
        }
    

    sigmas = {
        "STD accuracy": accuracy_std,
        "STD recall": recall_std,
        "STD precision": precision_std,
        "STD F-1": f1_std,
        "STD training time": training_time_std,
        "STD AUC": auc_std,
    }

    
    intervals = {
        "Interval accuracy": interval_acc,
        "Interval recall": interval_rec,
        "Interval precision": interval_pre,
        "Interval F-1": interval_f1,
        "Interval training time": interval_time,
        "Interval AUC": interval_auc,
        }
    
    return [averages, intervals, sigmas]

# Experiment with learning rate

In [40]:
with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    alad_list = []
    ae_list = []
    cae_list = []
    vae_list = []

    for rate in learing_rates:

        ae, cae, vae, alad = build_models(encoding_dim=ENCODING_DIM, learning_rate=rate)
        
        t = mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)
        alad_list.append(t[0])
        t = mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)
        ae_list.append(t[0])
        t = mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)
        cae_list.append(t[0])
        t = mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)
        vae_list.append(t[0])

### I'm using the recall as a parameter ot be optimized for the learning_rate search.

In [41]:
def find_best(method_list, variable_list):

    maxx = 0

    for index, value in enumerate(method_list):

        if value["Average recall"] > maxx:
            maxx = value["Average recall"]
            best_rate = variable_list[index]

    return str(best_rate)

print("the best rate for alad method is " + find_best(alad_list, learing_rates))
print("the best rate for ae method is " + find_best(ae_list, learing_rates))
print("the best rate for cae method is " + find_best(cae_list, learing_rates))
print("the best rate for vae method is " + find_best(vae_list, learing_rates))

the best rate for alad method is 0.001
the best rate for ae method is 0.05
the best rate for cae method is 0.001
the best rate for vae method is 0.05


In [ ]:
LEARNING_RATE = 0.001

# Experiment with encoding dimension

In [43]:
encoding_dims = list(range(2, round(INPUT_DIM / 2),  1))
encoding_dims = [2, 3, 5]

with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    alad_list = []
    ae_list = []
    cae_list = []
    vae_list = []

    for encoding_dim in encoding_dims:

        ae, cae, vae, alad = build_models(encoding_dim=encoding_dim, learning_rate=0.001)
        
        t = mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)
        alad_list.append(t[0])
        t = mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)
        ae_list.append(t[0])
        t = mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)
        cae_list.append(t[0])
        t = mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)
        vae_list.append(t[0])

In [45]:
print("the best encoding_dim for alad method is " + find_best(alad_list, encoding_dims))
print("the best encoding_dim for ae method is " + find_best(ae_list, encoding_dims))
print("the best encoding_dim for cae method is " + find_best(cae_list, encoding_dims))
print("the best encoding_dim for vae method is " + find_best(vae_list, encoding_dims))

the best encoding_dim for alad method is 2
the best encoding_dim for ae method is 5
the best encoding_dim for cae method is 2
the best encoding_dim for vae method is 2


In [ ]:
ENCODING_DIM = 2

# Experiment with other methods

In [46]:
from sklearn import svm

def SVM_predict(train_normal_data, x_test):
    
    clf = svm.OneClassSVM(nu=0.1, kernel="rbf", gamma=0.2)
    clf.fit(train_normal_data)
    # predict if the test instaces are anomalies or not
    #Returns -1 for outliers and 1 for inliers.
    y_pred = clf.predict(x_test)  
    y_pred = [1 if pred == -1 else 0 for pred in y_pred]
    
    return y_pred


In [47]:
from sklearn.ensemble import IsolationForest

def IF_predict(x_train, x_test):

    clf = IsolationForest(n_estimators=10000, max_samples='auto', contamination=0.2, random_state=42)
    clf.fit(x_train)
    y_pred = clf.predict(x_test)
    y_pred = [1 if pred == -1 else 0 for pred in y_pred]

    return y_pred


In [48]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score


def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    return accuracy, precision, recall, f1

In [49]:
def test_methods(function, alpha=0.95, iterations=2):
    #initialize all the containers for teh metrics to track
    list_fpr = []
    list_tpr = []
    list_auc = []

    #precision and recall will be used for 
    #the curve but also as general parameters 
    list_precision_curve = []
    list_recall_curve = []

    #classification metrics 
    list_accuracy = []
    list_precision = []
    list_recall = []
    list_f1 = []


    for _ in range(iterations):

        x_train, x_test, y_test = get_data()

        predictions = function(x_train, x_test)
        
        accuracy, precision, recall, f1 = calculate_metrics(y_test, predictions)

        fpr_ae, tpr_ae, _ = roc_curve(y_test, predictions)
        auc_ae = auc(fpr_ae, tpr_ae)
        list_fpr.append(fpr_ae)
        list_tpr.append(tpr_ae)
        

        precision_curve, recall_curve, _ = precision_recall_curve(y_test, predictions)
        list_precision_curve.append(precision_curve)
        list_recall_curve.append(recall_curve)


        list_accuracy.append(accuracy)
        list_precision.append(precision)
        list_recall.append(recall)
        list_f1.append(f1)
        list_auc.append(auc_ae)

    #roc_curve can return different sized arrays
    #this means that ther needs to be a custom function
    #to compute the mean 
    avg_fpr = avgNestedLists(list_fpr)
    avg_tpr = avgNestedLists(list_tpr)
    avg_precision_curve = avgNestedLists(list_precision_curve)
    avg_recall_curve = avgNestedLists(list_recall_curve)
    avg_acc = np.mean(list_accuracy)
    avg_rec = np.mean(list_recall)
    avg_pre = np.mean(list_precision)
    avg_f1 = np.mean(list_f1)
    avg_auc = np.mean(list_auc)


    #standard deviations
    accuracy_std = np.std(list_accuracy)
    precision_std = np.std(list_precision)
    recall_std = np.std(list_recall)
    f1_std = np.std(list_f1)
    auc_std = np.std(list_auc)

    #how to take the "mean curve"  
    interval_acc = stats.t.interval(alpha=alpha, loc=np.mean(list_accuracy), df=len(list_accuracy)-1, scale=stats.sem(list_accuracy))
    interval_rec = stats.t.interval(alpha=alpha, loc=np.mean(list_recall), df=len(list_recall)-1, scale=stats.sem(list_recall))
    interval_pre = stats.t.interval(alpha=alpha, loc=np.mean(list_precision), df=len(list_precision)-1, scale=stats.sem(list_precision))
    interval_f1 = stats.t.interval(alpha=alpha, loc=np.mean(list_f1), df=len(list_f1)-1, scale=stats.sem(list_f1))
    interval_auc = stats.t.interval(alpha=alpha, loc=np.mean(list_auc), df=len(list_auc)-1, scale=stats.sem(list_auc))




    averages = {
        "Average FPR": avg_fpr,
        "Average TPR": avg_tpr,
        "Average precision curve": avg_precision_curve,
        "Average recall curve": avg_recall_curve,
        "Average accuracy": avg_acc,
        "Average recall": avg_rec,
        "Average precision": avg_pre,
        "Average F-1": avg_f1,
        "Average AUC": avg_auc,
        }
    

    sigmas = {
        "STD accuracy": accuracy_std,
        "STD recall": recall_std,
        "STD precision": precision_std,
        "STD F-1": f1_std,
        "STD AUC": auc_std,
    }

    
    intervals = {
        "Interval accuracy": interval_acc,
        "Interval recall": interval_rec,
        "Interval precision": interval_pre,
        "Interval F-1": interval_f1,
        "Interval AUC": interval_auc,
        }
    
    return [averages, intervals, sigmas]

In [50]:

with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):
    ae, cae, vae, alad = build_models(encoding_dim=ENCODING_DIM, learning_rate=0.001)
        
    alad_std = mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)
    ae_std = mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)
    cae_std = mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)
    vae_std = mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)

    if_vals = test_methods(IF_predict, iterations =10)
    svm_vals = test_methods(SVM_predict, iterations =10)

KeyboardInterrupt: 